# 📈 Volume momentum — intraday backtest & optimize

**The idea.** A minute with far more volume than usual signals the arrival of new information or a large participant. "More than usual" is judged against the **time-of-day volume profile**: today's 09:35 volume is compared to the average of the last `profile_days` sessions at 09:35 (± `smooth_minutes` around it) — so the routinely heavy open and close minutes never trigger by themselves. Volume decides **when** to act; a **momentum indicator** — SMA(`fast`) − SMA(`slow`) — reads the current market condition and decides **which way**: momentum up → long, momentum down → short (both directions are always live, there is no long/short switch). Entries are blocked in the first `skip_open` / last `skip_close` minutes of each session, a fixed or trailing **stop loss** (`sl_pct`, `sl_trail`) caps every trade, and `eod_exit` forces flat on each day's final bar — nothing is held overnight. Full thesis in [README.md](./README.md).

The `data:` block deliberately loads only **~1 month of 1-minute bars** while this setup is being tested — widen `start` in [`config.yaml`](./config.yaml) once it looks sane (symbols without 1m history are skipped; backfill more via `notebooks/02_backfill.ipynb`). Every knob lives there too (defaults, sweep grid, optimization space) — this notebook only *drives* the shared engine: **load → overview → backtest → validate → sweep → optimize → boundary-check → verify**. The same steps work for every strategy in this repo.

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import backtest as bt, jobs, optimize as opt, strategies
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple import plots

strat = strategies.load("volume_momentum")
print("data :", strat.data)
print("params:", strat.params)

In [ ]:
d = strat.data
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]),
                         d["bar"], d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"panel: {panel.close.shape[0]:,} bars x {len(panel.symbols)} symbols "
      f"({panel.close.index[0]:%Y-%m-%d} → {panel.close.index[-1]:%Y-%m-%d})")

## 2 · Strategy overview — hyperparameters and search ranges

What each knob does, before any number crunching:

| param | role in the strategy | intuition |
|---|---|---|
| `fast` / `slow` | momentum SMA windows (bars) | their difference is the direction filter; `fast < slow` enforced |
| `threshold` | spike multiple vs expected volume | **the** knob: higher = rarer, cleaner events |
| `horizon` | bars held after entry | how long the volume burst is trusted to carry |
| `baseline` | `auto` / `sma` / `profile` | expected-volume model (`auto`: profile intraday, sma on 1d) |
| `profile_days` | profile lookback (sessions) | how many past days define "usual" volume at each minute |
| `smooth_minutes` | profile smoothing (± minutes) | 09:35 compares to 09:35±N — steadies thin 1-minute prints |
| `vol_window` | sma-baseline window (bars) | only used with `baseline: sma` |
| `skip_open` / `skip_close` | entry blackout (minutes) | no entries into the open/close chaos |
| `sl_pct` / `sl_trail` | stop loss: distance, fixed vs trailing | 0.02 = 2% of entry; trailing follows the best price |
| `eod_exit` | flat on each session's final bar | intraday means intraday — no overnight risk |

The table below shows, per parameter: the **default** used by the single
backtest (§3), the hand-picked **sweep grid** (§6), and the **Optuna search
range** (§7). A `—` means the stage keeps the default.

In [ ]:
import pandas as pd

def _fmt_space(spec):
    """One readable line for an optuna.space entry."""
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "default": pd.Series(strat.params),
    "sweep grid (§6)": pd.Series({k: ", ".join(map(str, v)) for k, v in strat.sweep.items()}),
    "optuna search range (§7)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
}).fillna("—")

## 3 · Single backtest with the config defaults

Engine rules (identical for every strategy): signals execute at the **next
bar's adjusted open**, realistic costs are always charged, every symbol trades
its own starting cash.

> **Capital model:** ONE shared \$10k wallet (engine cash-shared mode) — each entry deploys `cash_frac` of the *free* cash, so the book is never more than 100% invested. The repo-wide comparability rule; see `notebooks/04_compare.ipynb` for the cross-strategy scoreboard.

In [ ]:
import pandas as pd
pf = bt.run_strategy(strat, panel, settings=settings)
stats = bt.portfolio_stats(pf)
pd.Series(stats).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity, config defaults").show()

## 4 · Inspect a symbol — trades and indicator overlays

`inspect_symbol` is the microscope of this notebook, used twice: here on the
most-traded symbol with the *default* parameters, and again in §11 on the
*optimized* ones. How to read the figure:

* **Price panel** — ▲ blue = long entry, ▼ orange = short entry, ✕ = exit
  (time exit, stop-loss hit, or end-of-day flat); the entry→exit segment is
  green when the trade won, red when it lost.
* **Momentum panel** — the direction filter: blue above zero → long bias,
  red below zero → short bias.
* **Volume panel** — gray bars are ordinary volume; the solid line is the
  expected volume (the smoothed time-of-day profile of the last
  `profile_days` sessions), the dashed line the spike trigger (`threshold ×`
  expected); **amber dots mark the bars that cross it** — exactly the events
  the strategy trades at the next bar's open. Note the trigger line's daily
  U-shape: heavy open/close volume is *expected*, so it takes far more than a
  routine open to fire there.

In [ ]:
def inspect_symbol(sym, pf, params):
    """Deep-dive one symbol of a portfolio: annotated chart + its trade table."""
    all_trades = bt.trade_records(pf)
    merged = {**strat.params, **params, "bar": panel.bar, "tz": panel.tz}
    overlays = (strat.indicators(panel.close[sym], panel.volume[sym], **merged)
                if strat.indicators else None)
    plots.plot_symbol(panel.close[sym], panel.volume[sym], all_trades, symbol=sym,
                      overlays=overlays, equity=pf.value(group_by=False)[sym]).show()
    mine = all_trades[all_trades["symbol"] == sym]
    return mine.sort_values("entry_ts").reset_index(drop=True).round(3)

trades = bt.trade_records(pf)
busiest = trades["symbol"].value_counts().index[0]
inspect_symbol(busiest, pf, strat.params)

## 5 · Validate the volume-profile baseline

The detector is only as good as its expectation, so look at the raw material
before optimizing anything. This figure puts everything on one exchange-local
clock (log volume axis): **thin gray lines** are the last `profile_days`
sessions' volume by time of day (darker = more recent) — the inputs; the
**bold purple line** is the profile actually used on the target session
(their per-minute median, after ±`smooth_minutes` smoothing — taken straight
from `backtest.volume_profile`, the same call the strategy makes); **dashed
amber** is the `threshold ×` trigger; the **black line** is the target
session's own volume, with **amber dots** where it crosses the trigger.

Two things should be visible if the method works: the profile traces the
U-shape (heavy open/close volume is *expected*, so the trigger is high
exactly there and a routine open never fires), and genuine spikes stand clear
of the trigger in the quiet middle of the day. Change `PROFILE_SYMBOL` /
`PROFILE_DATE` below to inspect other names and sessions.

In [ ]:
PROFILE_SYMBOL = None   # ← None = the busiest symbol from §4
PROFILE_DATE = None     # ← e.g. "2026-06-24"; None = the last stored session
p = strat.params
psym = PROFILE_SYMBOL or busiest
pbase = bt.volume_profile(panel.volume[[psym]], panel.tz, days=p["profile_days"],
                          smooth=p["smooth_minutes"])[psym]
plots.plot_volume_profile(panel.volume[psym], pbase, panel.tz, days=p["profile_days"],
                          threshold=p["threshold"], symbol=psym, date=PROFILE_DATE).show()

## 6 · Grid sweep — a coarse map of the parameter space

Before optimizing, sweep the small hand-picked grid from `config.yaml`. The
ranked bars show the best single combos; the **heatmap grid** below them is the
multi-objective view — the same parameter pairs judged by Sharpe, total return
*and* breadth (`pct_profitable` = share of symbols that made money) at once, so
a region that pays on return but is carried by one name stands out immediately.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
print(f"{len(grid)} combos")
sweep_stats = bt.sweep(strat, panel, grid, settings=settings)
sweep_stats.sort_values("sharpe", ascending=False).head(8).round(3)

In [ ]:
plots.plot_sweep(sweep_stats).show()
plots.plot_heatmap_grid(
    sweep_stats,
    pairs=[("horizon", "threshold"), ("fast", "slow"), ("sl_pct", "threshold")],
    metrics=("sharpe", "total_return", "pct_profitable"),
).show()

## 7 · Optimize (resumable, runs outside the notebook)

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study
scores **`excess_cagr` (maximized) against `tracking_error` (minimized)**, both
relative to buy-and-hold, so the result is a *Pareto
front* — the set of parameter sets where improving one objective must hurt the
other. Every finished trial is checkpointed to `studies/{study}.log`;
**re-running this cell resumes** exactly where it stopped (Ctrl-C / kernel
restarts lose nothing).

While it runs, every finished trial streams one line below — its objectives,
the current front size, and its parameters — so you can watch the search move.
For a richer live view, copy the **`uvx optuna-dashboard …`** command from the
first output line into a second terminal: it serves parameter importances,
objective history and the front at http://127.0.0.1:8080 while the study runs.

In [ ]:
N_TRIALS = 200   # raise for a deeper search; re-runs accumulate into the same study
jobs.run_optimize("volume_momentum", n_trials=N_TRIALS)

## 8 · Read the study: Pareto front and the picked trial

Gray dots are all completed trials; the blue **staircase** is the exact
boundary of the dominated region (each labeled point is one Pareto-optimal
trial); the orange star is the trial `pick_best` selects under the weights
below. Hover any point for its full parameter set.

> **Note:** new trials are floor-pruned in-objective via `optuna.min_trades` in `config.yaml`; the admissibility filter below still guards older trials in resumed journals. The study name was bumped when the capital model changed to the shared wallet — earlier journals scored independent per-symbol books and are not comparable.

In [ ]:
study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)

# opt.PICK_WEIGHTS (balanced) is the repo's one preference — see optimize.py for why it
# is a constant and not a per-notebook literal. This cell used to say "full weight on
# Sharpe, half on raw return", which described the PRE-v5 objectives; the objectives are
# benchmark-relative (excess_cagr ↑ against tracking_error ↓) — see optimize.OBJECTIVES.
best = opt.pick_best(study)
plots.plot_pareto(front, everything, picked=best.number).show()
front.head(8).round(3)

## 9 · Boundary check — did the optimum hit the search-space edge?

A picked trial sitting **on the edge of a searched range** is a hint that the
true optimum may lie outside it. `optimize.boundary_report` compares the
picked parameters (and the whole Pareto front) with each parameter's `search:` range in
`config.yaml`: any numeric parameter within 5% of a bound — measured on the
searched scale, so `log: true` ranges (like `horizon`) are judged in log
space — is flagged with the bound to widen. Categorical parameters
(`sl_trail`) have no edges and are skipped.

If something is flagged: widen that bound in `config.yaml`, re-run §7 (the
same study name resumes, and the sampler explores the new region; rename the
study to restart clean), then re-read §8–§9 until the front sits inside the
box. `tsim optimize` prints the same check at the end of every CLI run.

In [ ]:
report = opt.boundary_report(best.params, strat.optuna["space"], front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
report

## 10 · Re-run the full backtest with the picked parameters

In [ ]:
print("picked params:", best.params)
pf_best = bt.run_strategy(strat, panel, best.params, settings=settings)
pd.Series(bt.portfolio_stats(pf_best)).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf_best, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity with picked parameters").show()

## 11 · Verification — per-symbol deep dive

Portfolio means can hide an edge carried by a single name, so verify the
picked parameters symbol by symbol. First the full per-symbol breakdown of the
optimized backtest; then set `SYMBOL` in the next cell to any name that looks
interesting (the big winner, the one loser, the one with suspiciously few
trades) and re-run it to get the same annotated chart and trade table as §4 —
entries, exits, direction, momentum and the volume spikes that triggered each
trade.

In [ ]:
bt.symbol_stats(pf_best).round(4)

In [ ]:
SYMBOL = None   # ← set to any symbol from the table above (e.g. "NVDA"); None = most-traded
sym = SYMBOL or bt.trade_records(pf_best)["symbol"].value_counts().index[0]
inspect_symbol(sym, pf_best, best.params)

## Read the result honestly

* Everything above is **in-sample**: the optimizer saw all the data it is being
  judged on. Before believing a parameter set, re-run it on a *different*
  universe and a held-out date range (edit `data:` in `config.yaml`).
* Prefer combos where **`pct_profitable` is high** — an edge carried by one
  symbol is usually noise.
* More trials on the same data = more chances to overfit. The Pareto front
  going up does not mean the *strategy* got better.

**Add your own strategy:** copy `strategies/_template/`, write `signals()`, fill
in `config.yaml` — this exact notebook flow works unchanged. See the README's
"How to add a strategy".